# 06 – Person Alternative Names

Esplorazione e data cleaning del dataset `person_alternate_names.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `person_mal_id` | ID univoco della persona su MyAnimeList |
| `alt_name` | Nome alternativo della persona |

## 1. Import e caricamento dati

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

In [2]:
df_alt = pd.read_csv('../datasets/person_alternate_names.csv')
print(f'Shape: {df_alt.shape}')
df_alt.info()
df_alt.head()

Shape: (20465, 2)
<class 'pandas.DataFrame'>
RangeIndex: 20465 entries, 0 to 20464
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   person_mal_id  20465 non-null  int64
 1   alt_name       20447 non-null  str  
dtypes: int64(1), str(1)
memory usage: 319.9 KB


,person_mal_id,alt_name
0,1,Seki Mondoya
1,1,門戸 開
2,1,Monto Hiraku
3,3,雪野五月
4,10,Kevin Hatcher


In [3]:
n_originale = len(df_alt)

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [4]:
mask_dup = df_alt.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_alt[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_alt.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_alt):,}')

Righe totali coinvolte in duplicazioni : 68
  → prime occorrenze mantenute         : 29
  → occorrenze extra rimosse           : 39

Righe prima della rimozione : 20,465
Righe dopo la rimozione     : 20,426


## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

In [5]:
analyze(df_alt['person_mal_id'])


════════════════════════════════════════════════════════════════════════════════
═══════════════════  🔬  SERIES ANALYSIS  —  'person_mal_id'  ═══════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    person_mal_id
  dtype                          int64
  Shape                          20,426
  Index range                    0 … 20464
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     20,426
  Non-null values                20,426  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          0  (0.00%)
  Positives                      20,426   (100.00%)
  Negatives                      0   (0.00%)
  Unique values                  12,376  (60.59%)

 📐 Descriptive Statistics
────────────────────────────────────────────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`. I duplicati sono attesi: una stessa persona può avere più nomi alternativi.

In [6]:
# Nessuna operazione richiesta
print(f'Null in person_mal_id  : {df_alt["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id (attesi) : {df_alt["person_mal_id"].duplicated().sum():,}')

Null in person_mal_id  : 0
Duplicati in person_mal_id (attesi) : 8,050


### 2.2 `alt_name`

In [7]:
analyze(df_alt['alt_name'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'alt_name'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    alt_name
  dtype                          str
  Shape                          20,426
  Index range                    0 … 20464
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     20,426
  Non-null values                20,408  [██████████████████████████████]  99.9%
  Null / NaN                     18  (0.09%)
  Empty strings                  0
  Unique values                  20,248  (99.13%)
  Duplicate values               160

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     1  chars
  Max le

**Pulizie necessarie:**
- 18 null residui → rimuovere le righe (una riga senza nome alternativo non ha significato)
- I nomi duplicati sono attesi: lo stesso nome può comparire per persone diverse

In [8]:
print(f'Null in alt_name prima della pulizia: {df_alt["alt_name"].isna().sum()}')

df_alt.dropna(subset=['alt_name'], inplace=True)

print(f'Null in alt_name dopo la pulizia    : {df_alt["alt_name"].isna().sum()}')
print(f'Righe dopo pulizia alt_name         : {len(df_alt):,}')

Null in alt_name prima della pulizia: 18
Null in alt_name dopo la pulizia    : 0
Righe dopo pulizia alt_name         : 20,408


### 2.3 Chiave primaria `(person_mal_id, alt_name)`

La chiave primaria naturale è la coppia `(person_mal_id, alt_name)`: verifichiamo che sia univoca dopo la pulizia.

In [9]:
n_pk_dup = df_alt.duplicated(subset=['person_mal_id', 'alt_name'], keep=False).sum()
print(f'Duplicati su (person_mal_id, alt_name): {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_alt.drop_duplicates(subset=['person_mal_id', 'alt_name'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_alt):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su (person_mal_id, alt_name): 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Data Cleaning

In [10]:
print('=== Riepilogo Dataset Pulito ===')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_alt):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_alt):>10,}')
print()
print('Dtypes finali:')
print(df_alt.dtypes)
print()
print('Valori mancanti residui:')
print(df_alt.isnull().sum())

=== Riepilogo Dataset Pulito ===
Righe originali      :     20,465
Righe dopo cleaning  :     20,408
Righe rimosse totali :         57

Dtypes finali:
person_mal_id    int64
alt_name           str
dtype: object

Valori mancanti residui:
person_mal_id    0
alt_name         0
dtype: int64


In [11]:
df_alt.head()

,person_mal_id,alt_name
0,1,Seki Mondoya
1,1,門戸 開
2,1,Monto Hiraku
3,3,雪野五月
4,10,Kevin Hatcher


## 4. Salvataggio dataset pulito

In [12]:
df_alt.to_csv('../datasets_cleaned/person_alternate_names_clean.csv', index=False)
print('Salvato: datasets_cleaned/person_alternate_names_clean.csv')

Salvato: datasets_cleaned/person_alternate_names_clean.csv
